In [ ]:
%pip install openai scikit-learn pandas

In [ ]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [ ]:
SYSTEM_PROMPT = """
You are a DASS-21 survey scoring assistant.
Your job is to extract answers for the DASS-21 based on a given conversation transcript.
The Depression Anxiety Stress Scales 21-item version (DASS-21) is a clinical assessment tool that measures the severity of depression, anxiety, and stress over the past week.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1  (Stress):      Hard to wind down or switch off
q6  (Stress):      Over-reacting to situations
q8  (Stress):      Using a lot of nervous energy
q11 (Stress):      Getting agitated
q12 (Stress):      Difficult to relax
q14 (Stress):      Impatient or intolerant
q18 (Stress):      Touchy or irritable
q2  (Anxiety):     Dryness of mouth
q4  (Anxiety):     Difficulty breathing
q7  (Anxiety):     Trembling
q9  (Anxiety):     Worried about panicking or embarrassing self
q15 (Anxiety):     Close to panic
q19 (Anxiety):     Aware of heart beating without physical exertion
q20 (Anxiety):     Scared without a clear reason
q3  (Depression):  Couldn't experience any positive feeling
q5  (Depression):  Difficult to work up initiative
q10 (Depression):  Nothing to look forward to
q13 (Depression):  Down-hearted or blue
q16 (Depression):  Unable to become enthusiastic
q17 (Depression):  Not worth much as a person
q21 (Depression):  Life felt meaningless

Scale mapping:
0 = never
1 = sometimes
2 = often
3 = almost always

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- Read each user response in order. The first user response answers q1, the second answers q6, the third answers q8, and so on following the order above.
- Score each response based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q21 key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q6": "0",
  "q8": "0",
  "q11": "0",
  "q12": "0",
  "q14": "0",
  "q18": "0",
  "q2": "0",
  "q4": "0",
  "q7": "0",
  "q9": "0",
  "q15": "0",
  "q19": "0",
  "q20": "0",
  "q3": "0",
  "q5": "0",
  "q10": "0",
  "q13": "0",
  "q16": "0",
  "q17": "0",
  "q21": "0"
}
"""
MODEL = 'qwen3:8b'

- Use a json dataset of DASS-21 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE

In [ ]:
# Load the DASS-21 dataset
import json

with open('dass_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

In [ ]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

In [ ]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [ ]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

In [ ]:
# Parse json scores
def parse_scores(raw_scores):
    parsed = []
    for s in scores:
    try:
        json_scores.append(json.loads(s))
    except json.JSONDecodeError:
        print(f'Failed to parse: {s}')
        json_scores.append({f'q{i}': '0' for i in range(1, 22)})
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)
    
    cols = [f'q{i}' for i in range(1, 22)]
    pred_df = pred_df[cols]
    expected_df = expected_df[cols]
    
    mse = mean_squared_error(
        convert_scores_to_array(expected_df),
        convert_scores_to_array(pred_df)
    )
    print(f'{name} MSE: {mse}')
    
    display(expected_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    ))

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')